In [0]:
# Notebook 02_kddcup_etl.ipynb
# Célula 1: Carregamento de Dados (Volume do Unity Catalog)

from pyspark.sql.functions import when, col, lit
from pyspark.sql.types import DoubleType, StringType, StructType, StructField
# VectorAssembler é usado na etapa de Transformação, mas pode ser importado aqui.
from pyspark.ml.feature import VectorAssembler

# --- 1.1. Definição do Caminho e Colunas ---

# CAMINHO DO VOLUME DO UNITY CATALOG (Refatorado)
FILE_PATH = "/Volumes/bigdata_anomaly_detection_kddcup99_catalogue/default/kdd_volume/kddcup.data.corrected" 

# Define os nomes das 41 features + 1 Target
COLUMN_NAMES = [f"f{i}" for i in range(1, 42)] + ["label"]

# --- 1.2. Carregamento Bruto (como strings) ---
try:
    print(f"Tentando carregar dados de: {FILE_PATH}")
    
    # O Spark lê o arquivo do Volume de forma distribuída
    df_raw = spark.read.csv(
        FILE_PATH, 
        header=False, 
        inferSchema=False, # Continua como False para evitar I/O desnecessário
        sep=','
    ).toDF(*COLUMN_NAMES)
    
    # A ação .count() força o Spark a ler o arquivo
    print(f"Sucesso! Registros carregados: {df_raw.count():,}")
    df_raw.printSchema()
    df_raw.show(5)

except Exception as e:
    print(f"ERRO: Não foi possível carregar o arquivo do Volume do UC em {FILE_PATH}.")
    print("Verifique se o cluster está ATIVO e se o caminho/permissões do Volume estão corretos.")
    # raise e

# Note: A variável df_raw agora está definida para a próxima célula.

In [0]:
# Etapa 2: Preparação do Target e Conversão de Tipos

# --- 2.1. Criação do Target Binário (is_anomaly) ---

# A classe 'normal.' é a classe majoritária (0). Tudo o mais é anomalia (1).
df_target = df_raw.withColumn(
    "is_anomaly", 
    when(col("label") == lit("normal."), 0).otherwise(1)
)

print("Contagem de Registros (0=Normal, 1=Anomalia):")
# O .show() força o cálculo distribuído
df_target.groupBy("is_anomaly").count().show()

# --- 2.2. Identificação e Conversão de Features Numéricas ---

# Features Categóricas (que DEVEM ser tratadas, mas ignoramos inicialmente): f2, f3, f4.
# Lista de colunas a serem usadas para o modelo (numéricas)
NUMERIC_FEATURES = [c for c in COLUMN_NAMES if c not in ["f2", "f3", "f4", "label"]]

# Converte as colunas selecionadas para o tipo Double (necessário para modelos ML)
df_clean = df_target
for c in NUMERIC_FEATURES:
    df_clean = df_clean.withColumn(c, col(c).cast(DoubleType()))

print("Schema após conversão numérica:")
df_clean.printSchema()

In [0]:
# --- CÉLULA 3: Divisão Treino/Teste ---

print("Etapa 3: Dividindo o dataset em Treino e Teste (70/30)...")

# O Spark faz a divisão de forma distribuída
df_train, df_test = df_clean.randomSplit([0.7, 0.3], seed=42)

print(f"Registros de Treino: {df_train.count():,}")
print(f"Registros de Teste: {df_test.count():,}")

In [0]:
# --- CÉLULA 4: Pipeline de Pré-processamento (StandardScaler e PCA) ---

from pyspark.ml.feature import StandardScaler, PCA, VectorAssembler
from pyspark.ml import Pipeline

print("\nEtapa 4: Configurando e Treinando o Pipeline de Pré-processamento...")

# --- 4.1. Configuração dos Estimadores/Transformadores ---

# 1. Vector Assembler (Necessário para o Scaler e PCA)
# Usa as features numéricas do df_clean (NUMERIC_FEATURES definido na Célula 2)
FINAL_FEATURES = [c for c in NUMERIC_FEATURES if c != "label"] 

assembler = VectorAssembler(
    inputCols=FINAL_FEATURES, 
    outputCol="vector_features"
)

# 2. StandardScaler (Treinado APENAS no conjunto de TREINO)
scaler = StandardScaler(
    inputCol="vector_features", 
    outputCol="scaled_features", 
    withStd=True, 
    withMean=False
)

# 3. PCA (Treinado APENAS no conjunto de TREINO)
N_COMPONENTS = 10 
pca = PCA(
    k=N_COMPONENTS, 
    inputCol="scaled_features", 
    outputCol="features" # Nome final da coluna de features
)

# --- 4.2. Criação e Treinamento do Pipeline ---

pipeline = Pipeline(stages=[assembler, scaler, pca])

# O pipeline.fit() é a etapa crítica que treina o scaler e o PCA *somente* no df_train
print("Treinando o Pipeline (Scaler e PCA) no conjunto de Treino...")
pipeline_model = pipeline.fit(df_train)

print("Pipeline treinado. Agora aplicando as transformações...")

In [0]:
# --- CÉLULA 5: Aplicação da Transformação e Salvamento ---

# 5.1. Aplicação da Transformação
# Aplica o modelo treinado (scaler e PCA) ao conjunto de TREINO
df_train_processed = pipeline_model.transform(df_train).select("features", "is_anomaly")

# Aplica o modelo treinado (scaler e PCA) ao conjunto de TESTE
df_test_processed = pipeline_model.transform(df_test).select("features", "is_anomaly")

# 5.2. Salvamento (Para uso no Notebook 03)

BASE_TABLE_NAME = "bigdata_anomaly_detection_kddcup99_catalogue.default"

# Salva o conjunto de Treino
TRAIN_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_train"
print(f"\nSalvando Tabela de TREINO: {TRAIN_TABLE_NAME}")
df_train_processed.write.mode("overwrite").saveAsTable(TRAIN_TABLE_NAME)

# Salva o conjunto de Teste
TEST_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_test"
print(f"Salvando Tabela de TESTE: {TEST_TABLE_NAME}")
df_test_processed.write.mode("overwrite").saveAsTable(TEST_TABLE_NAME)

print("\nPré-processamento concluído sem vazamento de dados.")
df_train_processed.printSchema()